# Previsao de Obitos Mensais no Brasil

Pipeline de previsao do numero de obitos mensais registrados no Sistema de
Informacao sobre Mortalidade (SIM/DATASUS), agregados por mes e unidade
federativa, usando 30 anos de historico (1996-2025).

O modelo usa sazonalidade, tendencia e historico recente para prever os obitos
do mes seguinte com erro medio de 2,8% do volume mensal.

---

## 1. Setup

Instale as dependencias e baixe a serie agregada (obitos/mes/UF) e o modelo ja
treinado, diretamente do GitHub Releases.

In [ ]:
!pip install -q pandas scikit-learn numpy matplotlib joblib lightgbm

In [ ]:
# Baixar serie agregada e modelo de previsao treinado
import os, urllib.request, warnings
warnings.filterwarnings('ignore')

os.makedirs('models', exist_ok=True)
urllib.request.urlretrieve(
    'https://github.com/Weversson/Projeto_IML/releases/download/v1.0/serie_obitos_uf.csv',
    'serie_obitos_uf.csv')
urllib.request.urlretrieve(
    'https://github.com/Weversson/Projeto_IML/releases/download/v1.0/previsao_obitos_uf_lgbm.pkl',
    'models/previsao_obitos_uf_lgbm.pkl')
print('Arquivos baixados.')

# Opcional: clonar o repositorio para acesso ao codigo completo
# !git clone https://github.com/Weversson/Projeto_IML.git

---

## 2. Serie historica

A serie agregada tem 9.420 pontos. Com 30 anos x 12 meses x 27 UFs seriam 9.720,
mas faltam os 12 meses de 2012 em 25 UFs (apenas SP e TO tem 2012); por isso o
grafico mostra uma queda brusca em 2012. A serie nacional mostra sazonalidade
anual clara e um pico de 207.106 obitos em marco de 2021, no auge da pandemia
de COVID-19.

In [ ]:
import pandas as pd, numpy as np

series = pd.read_csv('serie_obitos_uf.csv')
series['DATA'] = pd.to_datetime(series['ANO'].astype(str) + '-' + series['MES'].astype(str) + '-01')

brasil = series.groupby('DATA')['OBITOS'].sum()
print(f'Periodo: {brasil.index.min().date()} a {brasil.index.max().date()}')
print(f'Media mensal: {brasil.mean():,.0f} obitos')
print(f'Pico: {brasil.idxmax().date()} ({brasil.max():,.0f})')

import matplotlib.pyplot as plt
plt.figure(figsize=(13, 4))
plt.plot(brasil.index, brasil.values, color='#2563eb', lw=1.1)
plt.axvline(pd.Timestamp('2020-03-01'), color='#dc2626', ls=':', lw=1.4, label='Inicio COVID')
plt.title('Obitos mensais no Brasil (1996-2025)')
plt.ylabel('Obitos'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---

## 3. Features e previsao

Cada linha e um mes por UF. Features usadas:

- SENO/COS: sazonalidade anual (posicao do mes em um ciclo de 12 meses)
- TEND: tendencia linear por UF
- LAG_1, LAG_2, LAG_6, LAG_12: obitos de 1, 2, 6 e 12 meses atras
- ROLL_3, ROLL_12: medias moveis de 3 e 12 meses
- COVID, POS: indicadores dos periodos de pandemia e pos-pandemia

Baselines: prever cada mes como o mesmo mes do ano anterior (sazonal ingenuo) e
repetir o valor do mes anterior.

Tecnica: LightGBM global, um unico modelo para as 27 UFs com a UF como codigo
numerico do IBGE. Treino jan/1997-dez/2023, teste 2024-2025 (24 meses), sem
embaralhamento.

In [ ]:
# Layout de features identico ao treinamento
df = pd.read_csv('serie_obitos_uf.csv')
df['DATA'] = pd.to_datetime(df['ANO'].astype(str) + '-' + df['MES'].astype(str) + '-01')
df = df.sort_values(['UF','DATA']).reset_index(drop=True)  # ordem temporal dentro de cada UF

df['SENO'] = np.sin(2*np.pi*df['MES']/12)
df['COS']  = np.cos(2*np.pi*df['MES']/12)
df['TEND'] = df.groupby('UF').cumcount()
df['LAG_1']  = df.groupby('UF')['OBITOS'].shift(1)
df['LAG_2']  = df.groupby('UF')['OBITOS'].shift(2)
df['LAG_6']  = df.groupby('UF')['OBITOS'].shift(6)
df['LAG_12'] = df.groupby('UF')['OBITOS'].shift(12)
df['ROLL_3']  = df.groupby('UF')['OBITOS'].transform(lambda s: s.rolling(3).mean()).shift(1)
df['ROLL_12'] = df.groupby('UF')['OBITOS'].transform(lambda s: s.rolling(12).mean()).shift(1)
df['COVID'] = ((df.DATA >= '2020-03-01') & (df.DATA <= '2022-12-01')).astype(float)
df['POS'] = (df['DATA'] >= '2023-01-01').astype(float)
df['NAIVE'] = df.groupby('UF')['OBITOS'].transform(lambda s: s.shift(12))

fc = ['UF','SENO','COS','TEND','COVID','POS','LAG_1','LAG_2','LAG_6','LAG_12','ROLL_3','ROLL_12','MES']
df = df.dropna(subset=['NAIVE','LAG_1']).copy()

import joblib
lgbm = joblib.load('models/previsao_obitos_uf_lgbm.pkl')
df['PREV'] = lgbm.predict(df[fc])

# Avaliacao no periodo de teste (2024-2025)
te = df[df['DATA'] >= '2024-01-01']
real = te.groupby('DATA')['OBITOS'].sum()
prev = te.groupby('DATA')['PREV'].sum()
base = te.groupby('DATA')['NAIVE'].sum()

print(f'Baseline sazonal: MAE {abs(real-base).mean():,.0f} obitos/mes')
print(f'LightGBM:          MAE {abs(real-prev).mean():,.0f} obitos/mes')
print(f'Reducao de MAE:    {(1 - abs(real-prev).mean()/abs(real-base).mean())*100:.1f}%')

In [ ]:
# Grafico: real vs previsao (2024-2025)
hist = df[df['DATA'] >= '2018-01-01'].groupby('DATA')['OBITOS'].sum()
plt.figure(figsize=(13, 5))
plt.plot(hist.index, hist.values, color='#2563eb', lw=1.2, label='Real (historico)')
plt.plot(prev.index, prev.values, color='#16a34a', lw=2, marker='o', ms=3, label='Previsao LightGBM')
plt.plot(base.index, base.values, color='#f59e0b', lw=1.6, ls='--', label='Baseline sazonal')
plt.axvline(pd.Timestamp('2020-03-01'), color='#dc2626', ls=':', lw=1.4, label='Inicio COVID')
plt.title('Obitos mensais no Brasil: real vs previsao', fontsize=13)
plt.xlabel('Ano'); plt.ylabel('Obitos por mes')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Treinamento do modelo

Codigo que treina o LightGBM carregado acima. O treino usa os meses ate
dez/2023 e o teste fica com 2024-2025, sem embaralhamento. A UF entra como
codigo numerico do IBGE. A semente fixa (random_state=42) faz o treino gerar
sempre o mesmo modelo.

No final, a celula compara as previsoes do modelo treinado aqui com as do
modelo publicado no Release v1.0.

In [ ]:
# Treinamento do LightGBM global (mesmos parametros do modelo publicado)
import lightgbm as lgb

treino = df[df['DATA'] < '2024-01-01']
print(f'Treino: {len(treino):,} linhas ({treino.DATA.min():%m/%Y} a {treino.DATA.max():%m/%Y})')
print(f'Teste:  {len(te):,} linhas ({te.DATA.min():%m/%Y} a {te.DATA.max():%m/%Y})')

modelo = lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.05, num_leaves=127, max_depth=10,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.7,
    random_state=42, verbose=-1)
modelo.fit(treino[fc], treino['OBITOS'])
joblib.dump(modelo, 'models/previsao_obitos_uf_lgbm_treinado.pkl')

# Verificacao: o modelo treinado aqui gera as mesmas previsoes do modelo do Release?
df['PREV_NOVO'] = modelo.predict(df[fc])
print(f"Maior diferenca para o modelo do Release: {abs(df['PREV_NOVO'] - df['PREV']).max()}")
print(f"Previsoes identicas ao Release: {np.allclose(df['PREV_NOVO'], df['PREV'])}")

prev_novo = df[df['DATA'] >= '2024-01-01'].groupby('DATA')['PREV_NOVO'].sum()
print(f'MAE do modelo treinado: {abs(real - prev_novo).mean():,.0f} obitos/mes')

### Comparacao detalhada

Alem do baseline sazonal, comparamos com um baseline que repete o mes anterior,
que usa a mesma informacao do LAG_1. Tambem calculamos o erro por UF nos 24
meses de teste e a importancia de cada feature no modelo. Esta celula usa as
previsoes ja feitas acima e nao treina nada.

In [ ]:
# Serie nacional no teste (2024-2025): modelo contra as duas linhas de base
mes_anterior = te.groupby('DATA')['LAG_1'].sum()
for nome, p in [('Baseline sazonal', base), ('Baseline mes anterior', mes_anterior), ('LightGBM', prev)]:
    print(f'{nome:22s} MAE {abs(real - p).mean():6,.0f}   RMSE {np.sqrt(((real - p) ** 2).mean()):6,.0f}')
print(f'Reducao de MAE frente ao mes anterior: {(1 - abs(real - prev).mean() / abs(real - mes_anterior).mean()) * 100:.1f}%')

# Erro por UF: MAE de cada estado nos 24 meses de teste
SIGLAS = {11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO', 21: 'MA', 22: 'PI',
          23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES',
          33: 'RJ', 35: 'SP', 41: 'PR', 42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'}
por_uf = (te.assign(ERRO_LGBM=abs(te['OBITOS'] - te['PREV']), ERRO_BASE=abs(te['OBITOS'] - te['NAIVE']))
            .groupby('UF')
            .agg(OBITOS_MES=('OBITOS', 'mean'), MAE_LGBM=('ERRO_LGBM', 'mean'), MAE_BASE=('ERRO_BASE', 'mean')))
por_uf.index = por_uf.index.map(SIGLAS)
por_uf['ERRO_%'] = 100 * por_uf['MAE_LGBM'] / por_uf['OBITOS_MES']
vence = por_uf['MAE_LGBM'] < por_uf['MAE_BASE']
print(f'\nLightGBM vence o baseline sazonal em {vence.sum()} de {len(por_uf)} UFs. Perde em: {", ".join(por_uf.index[~vence])}')
print(f"Soma do MAE das UFs: {por_uf['MAE_LGBM'].sum():,.0f} obitos/mes (na soma nacional, erros entre UFs se compensam)")
print(por_uf.sort_values('OBITOS_MES', ascending=False).round(1).to_string())

# Importancia das features (gain: quanto cada uma reduziu o erro no total)
gain = pd.Series(lgbm.booster_.feature_importance('gain'), index=lgbm.booster_.feature_name())
print('\nImportancia das features (% do gain):')
print((100 * gain / gain.sum()).sort_values(ascending=False).round(1).to_string())

---

## 4. (Opcional) Reconstruir a serie a partir dos dados brutos

Os arquivos originais do DATASUS somam ~40 GB. A serie agregada ja esta
disponivel no release, mas para reproducao completa o codigo abaixo reconstroi
o arquivo serie_obitos_uf.csv. Descomente as linhas para executar.

In [ ]:
# Importante: baixa e processa ~40 GB. Use apenas se precisar da reproducao completa.

# --- 4.1 Baixar os ZIPs de 1996 a 2025 (2,4 GB comprimidos) ---
# import os, urllib.request
# BASE_URL = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/json"
# os.makedirs('data/datasus', exist_ok=True)
# for year in range(2025, 1995, -1):
#     url = f"{BASE_URL}/Mortalidade_Geral_{year}_json.zip"
#     dest = f"data/datasus/Mortalidade_Geral_{year}.zip"
#     if not os.path.exists(dest):
#         print(f'Baixando {year}...')
#         urllib.request.urlretrieve(url, dest)
#     else:
#         print(f'{year} ja existe.')

# --- 4.2 Extrair e concatenar particoes (menos de 30s por ano) ---
# import zipfile
# for year in range(1996, 2026):
#     z = zipfile.ZipFile(f"data/datasus/Mortalidade_Geral_{year}.zip")
#     parts = sorted(p for p in z.namelist() if p.endswith('.json'))
#     with open(f"data/datasus/parte_{year}.json", 'wb') as out:
#         out.write(b'[')
#         for i, p in enumerate(parts):
#             content = z.read(p).decode('utf-8', errors='ignore').strip()
#             if content.startswith('['): content = content[1:]
#             if content.endswith(']'):   content = content[:-1]
#             if i > 0: out.write(b',')
#             out.write(content.encode())
#         out.write(b']')

# --- 4.3 Agregar por (ano, mes, UF) ---
# import json, pandas as pd
# rows = []
# for year in range(1996, 2026):
#     with open(f"data/datasus/parte_{year}.json") as f:
#         data = json.load(f)
#     for d in data:
#         mun = d.get('CODMUNRES') or ''
#         dt  = d.get('DTOBITO') or ''
#         if len(mun) >= 2 and len(dt) >= 8:
#             mes = dt[2:4]
#             if mes.isdigit() and 1 <= int(mes) <= 12:
#                 rows.append((year, int(mes), mun[:2]))
#     print(f'{year}: {len(rows):,} acumulados')
# agg = pd.DataFrame(rows, columns=['ANO','MES','UF']).groupby(['ANO','MES','UF']).size().reset_index(name='OBITOS')
# agg.to_csv('serie_obitos_uf.csv', index=False)
# print(f"serie_obitos_uf.csv gerado: {agg.shape[0]:,} linhas, {agg['OBITOS'].sum():,} obitos")

---

## 5. Reproducao e modelos

O modelo treinado (LightGBM, ~2,3 MB) e a serie agregada (~140 KB) estao
publicados no GitHub Releases e foram baixados na secao 1. O uso do Release vem
da primeira versao do projeto, quando o modelo de classificacao tinha 170 MB e
nao cabia no git (limite de 100 MB por arquivo).

O codigo que treina o modelo esta na secao 3, celula "Treinamento do modelo".
Abaixo, exportamos as previsoes do periodo de teste para uma planilha.

In [ ]:
# Exportar previsoes do periodo de teste
pd.DataFrame({'DATA': real.index, 'REAL': real.values,
              'PREV': prev.values, 'BASELINE': base.values}
             ).to_csv('previsoes_2024_2025.csv', index=False)
print('Salvo em previsoes_2024_2025.csv')